# Task 1: Credit Scoring Model
This notebook implements a machine learning model to predict an individual's creditworthiness using financial data.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import time

## 1. Data Generation
Creating a synthetic financial dataset containing income, age, debts, and payment history.

In [2]:
def generate_credit_data(n_samples=2000):
    np.random.seed(42)
    income = np.random.normal(50000, 15000, n_samples)
    age = np.random.randint(18, 70, n_samples)
    employment_years = np.random.randint(0, 40, n_samples)
    debt = np.random.normal(10000, 5000, n_samples)
    payment_history = np.random.uniform(0, 1, n_samples)
    score = (0.3 * (income / 50000) + 0.4 * payment_history + 0.2 * (employment_years / 20) - 0.3 * (debt / 10000))
    noise = np.random.normal(0, 0.1, n_samples)
    target = ((score + noise) > 0.5).astype(int)
    return pd.DataFrame({'income': income, 'age': age, 'employment_years': employment_years, 'debt': debt, 'payment_history': payment_history, 'creditworthy': target})

df = generate_credit_data()
df.head()

,income,age,employment_years,debt,payment_history,creditworthy
0,57450.712295,47,30,12063.153115,0.234005,0
1,47926.035482,48,20,6987.991889,0.084388,0
2,59715.328072,55,33,10425.610286,0.259215,1
3,72845.447846,28,30,17974.767257,0.018928,0
4,46487.699379,52,24,10610.703570,0.035893,0


## 2. Feature Engineering and Preprocessing
We create the `debt_to_income` ratio and scale the features.

In [3]:
X = df.drop('creditworthy', axis=1)
y = df['creditworthy']
X['debt_to_income'] = X['debt'] / X['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Model Training
Using a Random Forest Classifier for high accuracy and efficiency.

In [4]:
start_train = time.time()
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print(f"Training Time: {time.time() - start_train:.4f}s")

Training Time: 0.1201s


## 4. Evaluation
Testing the model and calculating key metrics.

In [5]:
start_test = time.time()
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]
print(f"Testing Time: {time.time() - start_test:.4f}s")

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.2%}")
print(f"Precision: {precision_score(y_test, y_pred):.2%}")
print(f"Recall:    {recall_score(y_test, y_pred):.2%}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.2%}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Testing Time: 0.0574s
Accuracy:  88.25%
Precision: 84.96%
Recall:    80.71%
F1-Score:  82.78%
ROC-AUC:   0.9569

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.92      0.91       260
           1       0.85      0.81      0.83       140

    accuracy                           0.88       400
   macro avg       0.87      0.87      0.87       400
weighted avg       0.88      0.88      0.88       400

